# OmniField-JEPA on CIFAR-10 — Sparse Context, Continuous-Chunk Targets

A JEPA-style self-supervised version of the sparse CIFAR-10 protocol, with the OmniField backbone
inside a faithful i-JEPA training scaffold.

**Training data.** Each image has a fixed 40 % pixel pool (deterministic per-instance, same seed as
the reconstruction notebook). Per iteration we sample **M independently-anchored continuous chunks** as targets and let the
context be everything else in the pool — the natural 2D analogue of i-JEPA's `nenc=1, npred=4`
multi-block masking (`src/masks/multiblock.py:20-46` in the released i-JEPA repo):
- pick `M_PRED` random pool anchors;
- each target block = the `K_TGT` nearest *still-unused* pool members to its anchor;
- **context** = pool members not claimed by any target block (~`K_POOL − M_PRED·K_TGT`).

Because the M anchors are independent, target→context spatial distances vary widely from one
step to the next — sometimes adjacent to the context block, sometimes on the opposite side of
the image. This is the variability the user pointed out is essential to i-JEPA's recipe.

**Test data.** Only the fixed first-half of the pool (the 20 %) is exposed. No target chunk,
no full image — that is the entire input distribution for downstream evaluation.

**Architecture.**

| component | weights | inputs | output |
|---|---|---|---|
| `context_encoder` | trained | 20 % context tokens | latent set `g_ctx ∈ ℝ^(B,256,512)` |
| `target_encoder`  | EMA of context | full 40 % pool tokens | latent set `g_tgt ∈ ℝ^(B,256,512)` |
| `predictor`       | trained | `(g_ctx, γ(target_coords))` | `h_pred ∈ ℝ^(B,N_t,512)` |
| `target_readout`  | EMA of predictor | `(g_tgt, γ(target_coords))` | `h_tgt  ∈ ℝ^(B,N_t,512)` |

Loss = `smooth_l1(h_pred, stop_grad(layer_norm(h_tgt)))` — i-JEPA Eq. 1 of `src/train.py:295-313`.
EMA momentum follows i-JEPA's linear `0.996 → 1.0` schedule.

The `CoordReadout` predictor mirrors `VisionTransformerPredictor` in i-JEPA: project the latent
set down to a predictor bottleneck dim, add a learnable mask token to the projected target
coordinate encoding, concat with the bottlenecked context tokens, run a few transformer blocks,
slice off the target portion, project back to the encoder embedding dim.

**Anti-collapse hardening.** In addition to the i-JEPA scaffold (EMA target, stop-gradient, LayerNormed target, bottlenecked predictor), this notebook applies:
- **DINO-style EMA centering** of the target embedding (Caron et al., 2021): a running mean of `h_tgt` is subtracted before LayerNorm so the target distribution cannot drift toward a single point.
- **Slower EMA + predictor warmup**: `m: 0.999 → 1.0` (was 0.996) and the context-encoder is frozen for the first `WARMUP_STEPS=2000` updates so the predictor specializes against a quasi-static target before the encoder is allowed to drift.


In [ ]:
import os, math, ssl
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from einops import rearrange, repeat
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

ssl._create_default_https_context = ssl._create_unverified_context
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- shared experiment config ----
IMAGE_SIZE  = 32
N_PIX       = IMAGE_SIZE * IMAGE_SIZE          # 1024
FRAC_POOL   = 0.40                             # per-instance fixed 40% pool
K_POOL      = int(round(FRAC_POOL * N_PIX))    # 410
K_HALF      = K_POOL // 2                      # 205 — the fixed 20% test-time input

# Multi-block target sampling (mirrors i-JEPA `npred=4`, `nenc=1`).
M_PRED      = 4                                # number of target blocks per image
K_TGT       = 50                               # pool members per target block
N_TGT       = M_PRED * K_TGT                   # 200 total target tokens per image
K_CTX_TRAIN = K_POOL - N_TGT                   # 210 context tokens per image at train

CHANNELS    = 3
FOURIER_DIM = 96
POS_DIM     = FOURIER_DIM * 2                  # 192
INPUT_DIM   = CHANNELS + POS_DIM               # 195

LATENT_DIMS = (256, 384, 512)
NUM_LATENTS = (256, 256, 256)
EMBED_DIM   = LATENT_DIMS[-1]                  # 512 — encoder output dim
PRED_DIM    = 256                              # predictor / readout bottleneck (i-JEPA = 384/12-head)
PRED_LAYERS = 4
PRED_HEADS  = 8
PRED_HEAD_D = 32

# ---- JEPA-specific config ----
BATCH_SIZE  = 64
EPOCHS      = 30
LR          = 2e-4
EMA_START        = 0.999          # was 0.996 — slower target to avoid collapse on a short schedule
EMA_END          = 1.000
WARMUP_STEPS     = 2000           # encoder frozen, only predictor trains, for the first N steps
CENTER_MOMENTUM  = 0.9            # DINO-style EMA on target output mean (subtracted before LN)

POOL_SEED   = 12345
CKPT_DIR    = "./checkpoints_jepa"
RESUME      = True
PROBE_RESUME= True

print(f"Device: {DEVICE}")
print(f"K_POOL = {K_POOL}  K_HALF = {K_HALF} (test 20%)  "
      f"M_PRED = {M_PRED}  K_TGT = {K_TGT}  N_TGT = {N_TGT}  K_CTX_TRAIN = {K_CTX_TRAIN}")


In [ ]:
class JEPAChunkCIFAR10(Dataset):
    """Per-instance fixed 40% pool; **multi-block target / context-complement** partition.

    train=True : sample M_PRED disjoint continuous target chunks of K_TGT pool members each
                 (each anchored at an independent random pool index). The context is everything
                 else in the pool (= K_CTX_TRAIN ≈ K_POOL − M_PRED·K_TGT members). Across
                 iterations the target→context geometry varies widely, as in i-JEPA.
    train=False: only the fixed first-K_HALF of the pool is exposed (the 20% test-time input).
                 target/pool fields come back as empty tensors so downstream code cannot peek.
    """
    def __init__(self, base_dataset, image_size=IMAGE_SIZE, frac_pool=FRAC_POOL,
                 seed=POOL_SEED, train=True,
                 m_pred=M_PRED, k_tgt=K_TGT):
        self.base   = base_dataset
        self.N_pix  = image_size * image_size
        self.K_pool = int(round(frac_pool * self.N_pix))
        self.k_half = self.K_pool // 2
        self.m_pred = m_pred
        self.k_tgt  = k_tgt
        self.train  = train
        assert self.m_pred * self.k_tgt < self.K_pool, \
            f"M_PRED*K_TGT ({self.m_pred*self.k_tgt}) must be < K_pool ({self.K_pool})"

        rng = np.random.RandomState(seed)
        self.pool_idx = np.stack(
            [rng.permutation(self.N_pix)[:self.K_pool] for _ in range(len(base_dataset))],
            axis=0,
        ).astype(np.int64)

        ys, xs = torch.meshgrid(
            torch.linspace(-1.0, 1.0, image_size),
            torch.linspace(-1.0, 1.0, image_size),
            indexing='ij',
        )
        self.coords_all = torch.stack([ys, xs], dim=-1).view(self.N_pix, 2)

    def __len__(self):
        return len(self.base)

    def _sample_multi_target(self, pool_xy):
        """Returns (target_local_idx [N_TGT], ctx_local_idx [K_CTX_TRAIN]).

        Each of the M_PRED target anchors is sampled independently from the still-unclaimed
        pool members, giving spatially-varied target locations across iterations.
        """
        K = pool_xy.shape[0]
        used = np.zeros(K, dtype=bool)
        target_locals = []
        for _ in range(self.m_pred):
            avail = np.flatnonzero(~used)
            anchor_local = int(np.random.choice(avail))
            d2 = (pool_xy - pool_xy[anchor_local]).pow(2).sum(-1).numpy()
            d2[used] = np.inf
            order = np.argsort(d2, kind="stable")
            chosen = order[:self.k_tgt]
            used[chosen] = True
            target_locals.append(chosen)
        tgt_local = np.concatenate(target_locals).astype(np.int64)
        ctx_local = np.flatnonzero(~used).astype(np.int64)
        return tgt_local, ctx_local

    def __getitem__(self, idx):
        img, label = self.base[idx]
        pix_all    = img.permute(1, 2, 0).reshape(-1, 3)
        pool       = self.pool_idx[idx]
        pool_xy    = self.coords_all[pool]                # (K_pool, 2)

        if self.train:
            tgt_local, ctx_local = self._sample_multi_target(pool_xy)
            ctx_idx = pool[ctx_local]
            tgt_idx = pool[tgt_local]
            return {
                "ctx_pixels":   pix_all[ctx_idx],
                "ctx_coords":   self.coords_all[ctx_idx],
                "tgt_pixels":   pix_all[tgt_idx],
                "tgt_coords":   self.coords_all[tgt_idx],
                "pool_pixels":  pix_all[pool],
                "pool_coords":  self.coords_all[pool],
                "ctx_idx":      torch.from_numpy(ctx_idx),
                "label":        int(label),
            }
        else:
            ctx_idx = pool[:self.k_half]
            empty_px = torch.empty(0, 3)
            empty_xy = torch.empty(0, 2)
            return {
                "ctx_pixels":   pix_all[ctx_idx],
                "ctx_coords":   self.coords_all[ctx_idx],
                "tgt_pixels":   empty_px, "tgt_coords":  empty_xy,
                "pool_pixels":  empty_px, "pool_coords": empty_xy,
                "ctx_idx":      torch.from_numpy(ctx_idx),
                "label":        int(label),
            }


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3),
])
cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
cifar_test  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_ds = JEPAChunkCIFAR10(cifar_train, train=True)
test_ds  = JEPAChunkCIFAR10(cifar_test,  train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

batch = next(iter(train_loader))
print("train batch shapes:")
for k, v in batch.items():
    print(f"  {k:13s} {tuple(v.shape)}  {v.dtype}")
print(f"\n  ctx tokens / image: {batch['ctx_pixels'].shape[1]}  (expected {K_CTX_TRAIN})")
print(f"  tgt tokens / image: {batch['tgt_pixels'].shape[1]}  (expected {N_TGT} = {M_PRED}×{K_TGT})")


In [ ]:
# ---- OmniField backbone (shared with the reconstruction notebook) ----

class GaussianFourierFeatures(nn.Module):
    def __init__(self, in_features, mapping_size, scale=15.0):
        super().__init__()
        self.register_buffer('B', torch.randn((in_features, mapping_size)) * scale)
    def forward(self, coords):
        proj = coords @ self.B
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)


def get_sinusoidal_embeddings(n, d):
    assert d % 2 == 0
    position = torch.arange(n, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d, 2).float() * -(math.log(10000.0) / d))
    pe = torch.zeros(n, d)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class PreNorm(nn.Module):
    def __init__(self, dim, fn, context_dim=None):
        super().__init__()
        self.fn = fn
        self.norm = nn.LayerNorm(dim)
        self.norm_context = nn.LayerNorm(context_dim) if context_dim is not None else None
    def forward(self, x, **kw):
        x = self.norm(x)
        if self.norm_context is not None:
            kw['context'] = self.norm_context(kw['context'])
        return self.fn(x, **kw)


class GEGLU(nn.Module):
    def forward(self, x):
        x, gates = x.chunk(2, dim=-1)
        return x * F.gelu(gates)


class FeedForward(nn.Module):
    def __init__(self, dim, mult=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * mult * 2),
            GEGLU(),
            nn.Linear(dim * mult, dim),
        )
    def forward(self, x):
        return self.net(x)


class Attention(nn.Module):
    def __init__(self, query_dim, context_dim=None, heads=8, dim_head=64):
        super().__init__()
        inner = dim_head * heads
        context_dim = context_dim if context_dim is not None else query_dim
        self.scale, self.heads = dim_head ** -0.5, heads
        self.to_q   = nn.Linear(query_dim, inner, bias=False)
        self.to_kv  = nn.Linear(context_dim, inner * 2, bias=False)
        self.to_out = nn.Linear(inner, query_dim)
    def forward(self, x, context=None, mask=None):
        h = self.heads
        q = self.to_q(x)
        context = context if context is not None else x
        k, v = self.to_kv(context).chunk(2, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> (b h) n d', h=h), (q, k, v))
        sim  = torch.einsum('b i d, b j d -> b i j', q, k) * self.scale
        attn = sim.softmax(dim=-1)
        out  = torch.einsum('b i j, b j d -> b i d', attn, v)
        return self.to_out(rearrange(out, '(b h) n d -> b n (h d)', h=h))


class CascadedBlock(nn.Module):
    def __init__(self, dim, n_latents, input_dim, cross_heads, cross_dim_head,
                 self_heads, self_dim_head, residual_dim=None):
        super().__init__()
        self.latents       = nn.Parameter(get_sinusoidal_embeddings(n_latents, dim), requires_grad=True)
        self.cross_attn    = PreNorm(dim, Attention(dim, input_dim, heads=cross_heads,
                                                    dim_head=cross_dim_head), context_dim=input_dim)
        self.self_attn     = PreNorm(dim, Attention(dim, heads=self_heads, dim_head=self_dim_head))
        self.residual_proj = nn.Linear(residual_dim, dim) if residual_dim and residual_dim != dim else None
        self.ff            = PreNorm(dim, FeedForward(dim))
    def forward(self, context, residual=None):
        b = context.size(0)
        x = repeat(self.latents, 'n d -> b n d', b=b)
        x = self.cross_attn(x, context=context) + x
        if residual is not None:
            r = self.residual_proj(residual) if self.residual_proj is not None else residual
            x = x + r
        x = self.self_attn(x) + x
        x = self.ff(x) + x
        return x


class JEPAEncoder(nn.Module):
    """OmniField backbone (ICMR cascade + self-attn trunk). Outputs g, the
    latent set used in Eq. 1 of the paper (h^(L-1)). Same architecture as
    OmniFieldCIFAR in the reconstruction notebook, but without the
    pixel-decoder head."""
    def __init__(self, input_dim=INPUT_DIM,
                 latent_dims=LATENT_DIMS, num_latents=NUM_LATENTS,
                 cross_heads=4, cross_dim_head=128,
                 self_heads=8, self_dim_head=128,
                 num_trunk_layers=4):
        super().__init__()
        self.encoder_blocks = nn.ModuleList()
        prev = None
        for d, n in zip(latent_dims, num_latents):
            self.encoder_blocks.append(
                CascadedBlock(d, n, input_dim, cross_heads, cross_dim_head,
                              self_heads, self_dim_head, residual_dim=prev)
            )
            prev = d
        final = latent_dims[-1]
        self.trunk = nn.ModuleList([
            nn.ModuleList([
                PreNorm(final, Attention(final, heads=self_heads, dim_head=self_dim_head)),
                PreNorm(final, FeedForward(final)),
            ]) for _ in range(num_trunk_layers)
        ])
        self.latent_dim = final

    def forward(self, data):
        residual = None
        for block in self.encoder_blocks:
            residual = block(data, residual=residual)
        for sa, ff in self.trunk:
            residual = sa(residual) + residual
            residual = ff(residual) + residual
        return residual                 # (B, N_lat, latent_dim)  = g


class CoordReadout(nn.Module):
    """Predict per-coordinate embeddings from a latent set.

    One-to-one with i-JEPA's `VisionTransformerPredictor`:
      - `proj_g`     ↔ `predictor_embed`         (encoder_dim → predictor_dim)
      - `proj_query` ↔ `predictor_pos_embed`     (per-target positional info)
      - `mask_token` ↔ `mask_token`              (learnable, broadcast over targets)
      - `blocks`     ↔ `predictor_blocks`        (self-attn over [context, targets])
      - `proj_out`   ↔ `predictor_proj`          (predictor_dim → encoder_dim)
    The trained instance is the predictor; an EMA copy serves as the
    target-side readout.
    """
    def __init__(self, latent_dim=EMBED_DIM, predictor_dim=PRED_DIM,
                 query_in_dim=POS_DIM, num_layers=PRED_LAYERS,
                 heads=PRED_HEADS, dim_head=PRED_HEAD_D):
        super().__init__()
        self.proj_g     = nn.Linear(latent_dim, predictor_dim)
        self.proj_query = nn.Linear(query_in_dim, predictor_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.blocks = nn.ModuleList([
            nn.ModuleList([
                PreNorm(predictor_dim, Attention(predictor_dim, heads=heads, dim_head=dim_head)),
                PreNorm(predictor_dim, FeedForward(predictor_dim)),
            ]) for _ in range(num_layers)
        ])
        self.norm     = nn.LayerNorm(predictor_dim)
        self.proj_out = nn.Linear(predictor_dim, latent_dim)

    def forward(self, g, query_coords_enc):
        """g: (B, N_lat, latent_dim); query_coords_enc: (B, N_q, POS_DIM).
        Returns: (B, N_q, latent_dim)."""
        B, N_lat, _ = g.shape
        N_q         = query_coords_enc.shape[1]
        ctx_tok = self.proj_g(g)
        tgt_tok = self.proj_query(query_coords_enc) + self.mask_token   # broadcast
        x = torch.cat([ctx_tok, tgt_tok], dim=1)                        # (B, N_lat + N_q, D)
        for sa, ff in self.blocks:
            x = sa(x) + x
            x = ff(x) + x
        x = self.norm(x)
        return self.proj_out(x[:, N_lat:])                              # only target slice


In [ ]:
import copy

fourier_encoder = GaussianFourierFeatures(2, FOURIER_DIM, scale=15.0).to(DEVICE)
context_encoder = JEPAEncoder().to(DEVICE)
predictor       = CoordReadout(latent_dim=EMBED_DIM, predictor_dim=PRED_DIM).to(DEVICE)

# Target side: deep copies, no grad, EMA-updated each step.
target_encoder  = copy.deepcopy(context_encoder).to(DEVICE)
target_readout  = copy.deepcopy(predictor).to(DEVICE)
for p in target_encoder.parameters(): p.requires_grad_(False)
for p in target_readout.parameters(): p.requires_grad_(False)

n_enc  = sum(p.numel() for p in context_encoder.parameters())
n_pred = sum(p.numel() for p in predictor.parameters())
n_fe   = sum(p.numel() for p in fourier_encoder.parameters())
print(f"context_encoder: {n_enc/1e6:.2f}M  predictor: {n_pred/1e6:.2f}M  fourier: {n_fe/1e3:.1f}k")

# Optimizer covers ONLY context-side params + fourier (target side is EMA'd).
trainable_params = (
    list(context_encoder.parameters())
    + list(predictor.parameters())
    + list(fourier_encoder.parameters())
)
optimizer = AdamW(trainable_params, lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))


def encode_input(pixels, coords):
    return torch.cat([pixels, fourier_encoder(coords)], dim=-1)


def encode_query(coords):
    return fourier_encoder(coords)


# ---- DINO-style EMA centering for the target embedding ----
# Maintain a slow EMA of the per-feature mean of `h_tgt` (averaged across batch
# and target positions). Subtract this center from `h_tgt` before LayerNorm so
# the target distribution cannot drift toward a single point (a classic
# collapse mode). See Caron et al., DINO 2021.
class TargetCenter(nn.Module):
    def __init__(self, embed_dim, momentum=CENTER_MOMENTUM):
        super().__init__()
        self.momentum = momentum
        self.register_buffer("center", torch.zeros(1, 1, embed_dim))

    @torch.no_grad()
    def update(self, h):
        # h: (B, N, D) — batch & position-averaged mean per feature
        batch_center = h.mean(dim=(0, 1), keepdim=True)
        self.center.mul_(self.momentum).add_(batch_center, alpha=1.0 - self.momentum)

    def forward(self, h):
        return h - self.center


target_center = TargetCenter(EMBED_DIM, momentum=CENTER_MOMENTUM).to(DEVICE)


# EMA helpers — mirrors i-JEPA train.py:333-336
@torch.no_grad()
def ema_update(target_module, online_module, m):
    for p_q, p_k in zip(online_module.parameters(), target_module.parameters()):
        p_k.data.mul_(m).add_((1.0 - m) * p_q.detach())


def make_momentum_schedule(start, end, total_steps):
    """Linear schedule, matches i-JEPA `momentum_scheduler` in train.py:228."""
    def gen():
        for i in range(total_steps + 1):
            yield start + i * (end - start) / total_steps
    return gen()


In [ ]:
# ---- Checkpoint utilities (mirrors the reconstruction notebook) ----

os.makedirs(CKPT_DIR, exist_ok=True)
JEPA_LAST  = os.path.join(CKPT_DIR, "jepa_last.pt")
JEPA_BEST  = os.path.join(CKPT_DIR, "jepa_best.pt")
PROBE_LAST = os.path.join(CKPT_DIR, "probe_last.pt")
PROBE_BEST = os.path.join(CKPT_DIR, "probe_best.pt")


# Sweep stale tmp files
for _name in os.listdir(CKPT_DIR):
    if _name.startswith(".ckpt_") or _name.endswith(".pt.tmp"):
        try:    os.remove(os.path.join(CKPT_DIR, _name))
        except OSError: pass


def _save_atomic(state, path):
    tmp = path + ".tmp"
    try:
        torch.save(state, tmp)
        os.replace(tmp, path)
    except Exception:
        if os.path.exists(tmp):
            try: os.remove(tmp)
            except OSError: pass
        raise


def save_jepa_ckpt(path, *, epoch, train_loss, best_loss, momentum, history, global_step):
    _save_atomic({
        "epoch":            epoch,
        "context_encoder":  context_encoder.state_dict(),
        "predictor":        predictor.state_dict(),
        "target_encoder":   target_encoder.state_dict(),
        "target_readout":   target_readout.state_dict(),
        "fourier":          fourier_encoder.state_dict(),
        "optimizer":        optimizer.state_dict(),
        "scheduler":        scheduler.state_dict(),
        "target_center":    target_center.state_dict(),
        "train_loss":       train_loss,
        "best_loss":        best_loss,
        "momentum":         momentum,
        "global_step":      global_step,
        "history":          history,
    }, path)


def load_jepa_ckpt(path, map_location=DEVICE):
    s = torch.load(path, map_location=map_location, weights_only=False)
    context_encoder.load_state_dict(s["context_encoder"])
    predictor.load_state_dict(s["predictor"])
    target_encoder.load_state_dict(s["target_encoder"])
    target_readout.load_state_dict(s["target_readout"])
    fourier_encoder.load_state_dict(s["fourier"])
    optimizer.load_state_dict(s["optimizer"])
    scheduler.load_state_dict(s["scheduler"])
    if "target_center" in s:
        target_center.load_state_dict(s["target_center"])
    return s


def save_probe_ckpt(path, *, probe, epoch, train_acc, test_acc, best_acc):
    _save_atomic({
        "epoch":     epoch,
        "probe":     probe.state_dict(),
        "train_acc": train_acc,
        "test_acc":  test_acc,
        "best_acc":  best_acc,
    }, path)


def load_probe_ckpt(path, probe, map_location=DEVICE):
    s = torch.load(path, map_location=map_location, weights_only=False)
    probe.load_state_dict(s["probe"])
    return s


for p in (JEPA_BEST, JEPA_LAST, PROBE_BEST, PROBE_LAST):
    if os.path.exists(p):
        s = torch.load(p, map_location="cpu", weights_only=False)
        if "train_loss" in s:
            print(f"  found  {p}  @ epoch {s['epoch']}  train_loss={s['train_loss']:.4f}  m={s.get('momentum', float('nan')):.4f}")
        else:
            print(f"  found  {p}  @ epoch {s['epoch']}  test_acc={s.get('test_acc', float('nan')):.4f}")
    else:
        print(f"  none   {p}")


In [ ]:
# ---- JEPA training step (with DINO centering + predictor warmup) ----

def jepa_loss(batch):
    ctx_p  = batch['ctx_pixels'].to(DEVICE)
    ctx_c  = batch['ctx_coords'].to(DEVICE)
    tgt_c  = batch['tgt_coords'].to(DEVICE)
    pool_p = batch['pool_pixels'].to(DEVICE)
    pool_c = batch['pool_coords'].to(DEVICE)

    ctx_data  = encode_input(ctx_p,  ctx_c)
    pool_data = encode_input(pool_p, pool_c)
    tgt_q     = encode_query(tgt_c)

    # ---- Target side (no grad, centered, layer-normed, stop-grad) ----
    with torch.no_grad():
        g_tgt = target_encoder(pool_data)
        h_tgt = target_readout(g_tgt, tgt_q)            # (B, N_t, D)
        target_center.update(h_tgt)                      # EMA-update center first
        h_tgt = target_center(h_tgt)                     # subtract center
        h_tgt = F.layer_norm(h_tgt, (h_tgt.size(-1),))   # then per-token LN

    # ---- Context side ----
    g_ctx  = context_encoder(ctx_data)
    h_pred = predictor(g_ctx, tgt_q)

    return F.smooth_l1_loss(h_pred, h_tgt)


def _set_encoder_trainable(flag: bool):
    """Toggle context_encoder param gradients. Used to implement the predictor
    warmup: for the first WARMUP_STEPS, only the predictor (+ target_center
    bookkeeping) update."""
    for p in context_encoder.parameters():
        p.requires_grad_(flag)
        if not flag:
            p.grad = None   # drop any stale grads so optimizer ignores them


# ---- Training loop with EMA + checkpointing ----
history     = {"train_loss": [], "momentum": []}
start_epoch = 1
best_loss   = float("inf")
total_steps = EPOCHS * len(train_loader)
momentum_gen = make_momentum_schedule(EMA_START, EMA_END, total_steps)
global_step  = 0

if RESUME and os.path.exists(JEPA_LAST):
    s = load_jepa_ckpt(JEPA_LAST)
    start_epoch = s["epoch"] + 1
    history     = s["history"]
    best_loss   = s.get("best_loss", float("inf"))
    global_step = s.get("global_step", (start_epoch - 1) * len(train_loader))
    if os.path.exists(JEPA_BEST):
        b = torch.load(JEPA_BEST, map_location="cpu", weights_only=False)
        best_loss = min(best_loss, b["train_loss"])
    # advance momentum generator to match
    for _ in range(global_step):
        try: next(momentum_gen)
        except StopIteration: break
    print(f"Resumed JEPA from {JEPA_LAST} @ epoch {s['epoch']} step {global_step} (best_loss={best_loss:.4f})")
else:
    print(f"Starting JEPA fresh — no resume.")

# Set initial encoder trainability based on whether we're past warmup
_set_encoder_trainable(global_step >= WARMUP_STEPS)
if global_step < WARMUP_STEPS:
    print(f"Predictor warmup active: context_encoder frozen for {WARMUP_STEPS - global_step} more steps.")

epoch = start_epoch - 1
m = EMA_START
try:
    for epoch in range(start_epoch, EPOCHS + 1):
        context_encoder.train(); predictor.train(); fourier_encoder.train()
        epoch_loss, steps = 0.0, 0
        for batch in train_loader:
            loss = jepa_loss(batch)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()
            scheduler.step()

            # EMA update — target ← m * target + (1-m) * online
            try:    m = next(momentum_gen)
            except StopIteration: m = EMA_END
            ema_update(target_encoder, context_encoder, m)
            ema_update(target_readout, predictor,        m)

            global_step += 1
            # Unfreeze context_encoder once warmup is over
            if global_step == WARMUP_STEPS:
                _set_encoder_trainable(True)
                print(f"  [step {global_step}] predictor warmup complete — context_encoder unfrozen.")

            epoch_loss += loss.item(); steps += 1

        avg_loss = epoch_loss / max(steps, 1)
        history["train_loss"].append(avg_loss)
        history["momentum"].append(m)

        improved = avg_loss < best_loss
        if improved:
            best_loss = avg_loss
            save_jepa_ckpt(JEPA_BEST, epoch=epoch, train_loss=avg_loss,
                           best_loss=best_loss, momentum=m, history=history,
                           global_step=global_step)
        save_jepa_ckpt(JEPA_LAST, epoch=epoch, train_loss=avg_loss,
                       best_loss=best_loss, momentum=m, history=history,
                       global_step=global_step)

        center_norm = target_center.center.norm().item()
        tag = "  ★ new best (saved jepa_best.pt)" if improved else ""
        print(f"[Epoch {epoch:02d}/{EPOCHS}] jepa_loss={avg_loss:.4f}  "
              f"m={m:.4f}  ||center||={center_norm:.3f}  "
              f"lr={scheduler.get_last_lr()[0]:.2e}{tag}")

    print("\nDone.")

except KeyboardInterrupt:
    print(f"\nInterrupted. Last completed epoch: {epoch}. "
          f"Checkpoints: {JEPA_LAST} (epoch {epoch}), {JEPA_BEST} (best so far).")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ep = range(1, len(history['train_loss']) + 1)
ax.plot(ep, history['train_loss'], label='JEPA smooth-L1 loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_yscale('log')
ax.set_title('OmniField-JEPA — context-target chunk pretraining')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
ax2 = ax.twinx()
ax2.plot(ep, history['momentum'], color='tab:orange', alpha=0.5, label='EMA momentum')
ax2.set_ylabel('EMA m')
plt.tight_layout(); plt.show()


## Linear probe — classify CIFAR-10 from the JEPA context encoder

At test time **only the fixed 20 %** is visible (no target chunk, no full image). We freeze the
trained `context_encoder` and `fourier_encoder`, mean-pool the latent set to get `z = mean(g, dim=1)`
(the OmniField global feature), and fit a linear classifier on top.

During probe training the context-encoder input is a randomly-sampled *training* context chunk
(20 % of the pool); during probe eval the encoder sees the *fixed* test-side 20 %. This matches
the constraint that 20 % is all we have at test.


In [ ]:
class LinearProbe(nn.Module):
    def __init__(self, in_dim=EMBED_DIM, n_classes=10):
        super().__init__()
        self.fc = nn.Linear(in_dim, n_classes)
    def forward(self, z):
        return self.fc(z)


@torch.no_grad()
def extract_z(pixels, coords):
    """Frozen context_encoder forward → mean-pooled global feature z."""
    context_encoder.eval(); fourier_encoder.eval()
    g = context_encoder(encode_input(pixels, coords))   # (B, N_lat, EMBED_DIM)
    return g.mean(dim=1)                                # (B, EMBED_DIM)


def train_linear_probe(epochs=10, lr=1e-3, weight_decay=1e-4):
    probe = LinearProbe(in_dim=EMBED_DIM, n_classes=10).to(DEVICE)
    opt   = AdamW(probe.parameters(), lr=lr, weight_decay=weight_decay)
    ce    = nn.CrossEntropyLoss()

    # Freeze encoder + fourier params
    for p in context_encoder.parameters(): p.requires_grad_(False)
    for p in fourier_encoder.parameters(): p.requires_grad_(False)

    start_epoch = 1
    best_acc    = -1.0
    if PROBE_RESUME and os.path.exists(PROBE_LAST):
        s = load_probe_ckpt(PROBE_LAST, probe)
        start_epoch = s["epoch"] + 1
        best_acc    = s.get("best_acc", s.get("test_acc", -1.0))
        if os.path.exists(PROBE_BEST):
            b = torch.load(PROBE_BEST, map_location="cpu", weights_only=False)
            best_acc = max(best_acc, b.get("test_acc", -1.0))
        print(f"Resumed probe from {PROBE_LAST} @ epoch {s['epoch']}  best so far: {best_acc:.4f}")

    @torch.no_grad()
    def _eval():
        probe.eval()
        correct = total = 0
        for b in test_loader:
            ctx_p = b['ctx_pixels'].to(DEVICE)
            ctx_c = b['ctx_coords'].to(DEVICE)
            y     = b['label'].to(DEVICE)
            z     = extract_z(ctx_p, ctx_c)
            correct += (probe(z).argmax(-1) == y).sum().item()
            total   += y.size(0)
        return correct / total

    epoch = start_epoch - 1
    try:
        for epoch in range(start_epoch, epochs + 1):
            probe.train()
            run_loss = correct = total = 0
            for b in train_loader:
                ctx_p = b['ctx_pixels'].to(DEVICE)
                ctx_c = b['ctx_coords'].to(DEVICE)
                y     = b['label'].to(DEVICE)
                z     = extract_z(ctx_p, ctx_c)
                logits = probe(z); loss = ce(logits, y)
                opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
                run_loss += loss.item() * y.size(0)
                correct  += (logits.argmax(-1) == y).sum().item()
                total    += y.size(0)

            train_acc = correct / total
            test_acc  = _eval()
            improved  = test_acc > best_acc
            if improved:
                best_acc = test_acc
                save_probe_ckpt(PROBE_BEST, probe=probe, epoch=epoch,
                                train_acc=train_acc, test_acc=test_acc, best_acc=best_acc)
            save_probe_ckpt(PROBE_LAST, probe=probe, epoch=epoch,
                            train_acc=train_acc, test_acc=test_acc, best_acc=best_acc)

            tag = "  ★ new best (saved probe_best.pt)" if improved else ""
            print(f"[probe ep {epoch:02d}/{epochs}] loss={run_loss/total:.4f}  "
                  f"train_acc={train_acc:.4f}  test_acc(20%)={test_acc:.4f}{tag}")
    except KeyboardInterrupt:
        print(f"\nProbe training interrupted. Last completed epoch: {epoch}. "
              f"Checkpoints: {PROBE_LAST}, {PROBE_BEST}.")

    return probe


probe = train_linear_probe(epochs=10, lr=1e-3)
